**(# Predictive EDA)**

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px



In [4]:
banknifty = pd.read_csv(
    "../data/processed/bank_nifty_daily_amended.csv"
)

stocks = pd.read_csv(
    "../data/processed/constituent_stocks_data.csv"
)
event_window = pd.read_csv(
    "../data/processed/earnings_event_window.csv"
)
weights = pd.read_csv(
    "../data/processed/daily_weight_reconstruction_new.csv"
)
earnings=pd.read_csv("../data/processed/earnings.csv")


banknifty["date"] = pd.to_datetime(banknifty["date"])
stocks["date"] = pd.to_datetime(stocks["date"])
earnings["date"] = pd.to_datetime(earnings["date"])
weights["date"] = pd.to_datetime(weights["date"])


print("Bank Nifty:", banknifty.shape)
print("Stocks:", stocks.shape)
print("Earnings:", earnings.shape)
print("Weights:", weights.shape)

print("\nBank Nifty columns:")
print(banknifty.columns.tolist())

print("\nStocks columns:")
print(stocks.columns.tolist())

print("\nEarnings columns:")
print(earnings.columns.tolist())

print("\nWeights columns:")
print(weights.columns.tolist())

Bank Nifty: (908, 5)
Stocks: (13530, 7)
Earnings: (240, 11)
Weights: (11152, 4)

Bank Nifty columns:
['date', 'open', 'high', 'low', 'close']

Stocks columns:
['date', 'symbol', 'open', 'high', 'low', 'close', 'volume']

Earnings columns:
['Unnamed: 0', 'symbol', 'date', 'period_end', 'eps_actual', 'eps_forecast', 'revenue_actual', 'revenue_forecast', 'eps_surprise', 'revenue_surprise', 'reaction']

Weights columns:
['date', 'symbol', 'weight', 'weight_source']


In [7]:
# ============================================================
# CELL 2 — DATA CLEANING & DERIVED VARIABLES
# ============================================================

# ------------------------------------------------------------
# 1. Bank Nifty
# ------------------------------------------------------------

banknifty = banknifty.sort_values("date").reset_index(drop=True)

banknifty["daily_return_pct"] = (
    banknifty["close"].pct_change() * 100
)

banknifty["intraday_range_pct"] = (
    (banknifty["high"] - banknifty["low"])
    / banknifty["open"]
    * 100
)


# ------------------------------------------------------------
# 2. Constituent stocks
# ------------------------------------------------------------

stocks = (
    stocks
    .sort_values(["symbol", "date"])
    .reset_index(drop=True)
)

stocks["daily_return_pct"] = (
    stocks
    .groupby("symbol")["close"]
    .pct_change()
    * 100
)

stocks["intraday_range_pct"] = (
    (stocks["high"] - stocks["low"])
    / stocks["open"]
    * 100
)


# ------------------------------------------------------------
# 3. Earnings
# ------------------------------------------------------------

# Remove CSV index column.
earnings = earnings.drop(
    columns=["Unnamed: 0"],
    errors="ignore"
)

earnings = earnings.sort_values(
    ["symbol", "date"]
).reset_index(drop=True)

earnings["date"] = pd.to_datetime(
    earnings["date"]
)

earnings["period_end"] = pd.to_datetime(
    earnings["period_end"],
    errors="coerce"
)


# ------------------------------------------------------------
# 4. Convert numeric earnings fields
# ------------------------------------------------------------

numeric_earnings_cols = [
    "eps_actual",
    "eps_forecast",
    "revenue_actual",
    "revenue_forecast",
    "eps_surprise",
    "revenue_surprise",
    "reaction",
]

for col in numeric_earnings_cols:
    earnings[col] = pd.to_numeric(
        earnings[col],
        errors="coerce"
    )


# ------------------------------------------------------------
# 5. Weights
# ------------------------------------------------------------

weights = (
    weights
    .sort_values(["date", "symbol"])
    .reset_index(drop=True)
)

weights["weight"] = pd.to_numeric(
    weights["weight"],
    errors="coerce"
)


# ------------------------------------------------------------
# 6. Basic validation
# ------------------------------------------------------------

print("=" * 70)
print("CLEAN DATASET SUMMARY")
print("=" * 70)

print(f"Bank Nifty rows:       {len(banknifty):,}")
print(f"Stock rows:            {len(stocks):,}")
print(f"Earnings rows:         {len(earnings):,}")
print(f"Weight rows:           {len(weights):,}")

print("\nBank Nifty date range:")
print(
    banknifty["date"].min().date(),
    "→",
    banknifty["date"].max().date()
)

print("\nStock date range:")
print(
    stocks["date"].min().date(),
    "→",
    stocks["date"].max().date()
)

print("\nEarnings date range:")
print(
    earnings["date"].min().date(),
    "→",
    earnings["date"].max().date()
)

print("\nConstituents:", stocks["symbol"].nunique())

print("\nEarnings per constituent:")
print(
    earnings["symbol"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nMissing values in earnings:")
print(
    earnings[
        [
            "eps_actual",
            "eps_forecast",
            "revenue_actual",
            "revenue_forecast",
            "eps_surprise",
            "revenue_surprise",
            "reaction",
        ]
    ]
    .isna()
    .sum()
)

CLEAN DATASET SUMMARY
Bank Nifty rows:       908
Stock rows:            13,530
Earnings rows:         240
Weight rows:           11,152

Bank Nifty date range:
2023-01-02 → 2026-08-31

Stock date range:
2023-01-02 → 2026-08-31

Earnings date range:
2023-01-14 → 2026-11-05

Constituents: 15

Earnings per constituent:
symbol
AUBANK        16
AXISBANK      16
BANDHANBNK    16
BANKBARODA    16
CANBK         16
FEDERALBNK    16
HDFCBANK      16
ICICIBANK     16
IDFCFIRSTB    16
INDUSINDBK    16
KOTAKBANK     16
PNB           16
SBIN          16
UNIONBANK     16
YESBANK       16

Missing values in earnings:
eps_actual          15
eps_forecast         5
revenue_actual      15
revenue_forecast     5
eps_surprise        16
revenue_surprise    15
reaction            15
dtype: int64


In [8]:
# ============================================================
# CELL 3 — EVENT CALENDAR & T0 MAPPING
# ============================================================

# ------------------------------------------------------------
# 1. Keep only earnings events relevant to the Bank Nifty
#    study period.
# ------------------------------------------------------------

study_start = banknifty["date"].min()
study_end = banknifty["date"].max()

earnings_study = earnings[
    earnings["date"].between(study_start, study_end)
].copy()

print("=" * 70)
print("EARNINGS EVENTS INSIDE STUDY PERIOD")
print("=" * 70)

print("Raw earnings events:", len(earnings))
print("Events in study period:", len(earnings_study))
print(
    "Excluded future events:",
    len(earnings) - len(earnings_study)
)


# ------------------------------------------------------------
# 2. Identify whether earnings date itself was a trading day.
# ------------------------------------------------------------

trading_dates = set(banknifty["date"])

earnings_study["event_was_trading_day"] = (
    earnings_study["date"].isin(trading_dates)
)


# ------------------------------------------------------------
# 3. Map every earnings event to T0.
#
# T0 = earnings date if it is a trading day.
# Otherwise T0 = first trading day AFTER the earnings date.
# ------------------------------------------------------------

trading_dates_sorted = np.array(
    sorted(trading_dates),
    dtype="datetime64[ns]"
)

earnings_study["t0_date"] = earnings_study["date"].apply(
    lambda x: pd.Timestamp(
        trading_dates_sorted[
            np.searchsorted(
                trading_dates_sorted,
                np.datetime64(x),
                side="left"
            )
        ]
    )
)


# ------------------------------------------------------------
# 4. Event identifier
# ------------------------------------------------------------

earnings_study["event_id"] = (
    earnings_study["symbol"].astype(str)
    + "_"
    + earnings_study["date"].dt.strftime("%Y%m%d")
)


# ------------------------------------------------------------
# 5. T0 statistics
# ------------------------------------------------------------

total_events = len(earnings_study)

same_day_events = (
    earnings_study["event_was_trading_day"].sum()
)

shifted_events = (
    total_events - same_day_events
)

same_day_pct = (
    same_day_events / total_events * 100
)

shifted_pct = (
    shifted_events / total_events * 100
)

print("\n" + "=" * 70)
print("T0 AVAILABILITY")
print("=" * 70)

print(f"Total earnings events:        {total_events}")
print(
    f"Earnings on trading day:      "
    f"{same_day_events} ({same_day_pct:.2f}%)"
)
print(
    f"Earnings on non-trading day:  "
    f"{shifted_events} ({shifted_pct:.2f}%)"
)


# ------------------------------------------------------------
# 6. Check how many UNIQUE T0 dates exist.
#
# This is more important than event count because multiple
# banks can have earnings on the same T0.
# ------------------------------------------------------------

unique_t0_dates = (
    earnings_study["t0_date"].nunique()
)

event_per_t0 = (
    earnings_study
    .groupby("t0_date")
    .size()
)

print("\n" + "=" * 70)
print("UNIQUE T0 DATES")
print("=" * 70)

print(
    f"Unique T0 dates:              {unique_t0_dates}"
)

print(
    f"Average events per T0 date:   "
    f"{event_per_t0.mean():.2f}"
)

print(
    f"Maximum events on one T0:     "
    f"{event_per_t0.max()}"
)


# ------------------------------------------------------------
# 7. Distribution of event clustering
# ------------------------------------------------------------

print("\nEvents per T0 date:")
print(
    event_per_t0
    .value_counts()
    .sort_index()
    .rename("number_of_t0_dates")
    .to_string()
)


# ------------------------------------------------------------
# 8. How many events share their T0 with another event?
# ------------------------------------------------------------

shared_t0_events = (
    earnings_study
    .groupby("t0_date")["event_id"]
    .transform("count")
    > 1
)

print("\n" + "=" * 70)
print("EVENT CLUSTERING")
print("=" * 70)

print(
    f"Events sharing T0 with another event: "
    f"{shared_t0_events.sum()}"
)

print(
    f"Percentage of events sharing T0: "
    f"{shared_t0_events.mean() * 100:.2f}%"
)


# ------------------------------------------------------------
# 9. Store the clean event calendar
# ------------------------------------------------------------

event_calendar = earnings_study[
    [
        "event_id",
        "symbol",
        "date",
        "t0_date",
        "event_was_trading_day",
        "period_end",
        "eps_actual",
        "eps_forecast",
        "revenue_actual",
        "revenue_forecast",
        "eps_surprise",
        "revenue_surprise",
        "reaction",
    ]
].copy()


# ------------------------------------------------------------
# 10. Sanity checks
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SANITY CHECKS")
print("=" * 70)

print(
    "Duplicate event IDs:",
    event_calendar["event_id"].duplicated().sum()
)

print(
    "T0 dates outside Bank Nifty data:",
    (
        ~event_calendar["t0_date"].isin(
            banknifty["date"]
        )
    ).sum()
)

print(
    "Events with T0 before earnings date:",
    (
        event_calendar["t0_date"]
        < event_calendar["date"]
    ).sum()
)

print("\nEvent calendar created:", event_calendar.shape)

EARNINGS EVENTS INSIDE STUDY PERIOD
Raw earnings events: 240
Events in study period: 225
Excluded future events: 15

T0 AVAILABILITY
Total earnings events:        225
Earnings on trading day:      149 (66.22%)
Earnings on non-trading day:  76 (33.78%)

UNIQUE T0 DATES
Unique T0 dates:              136
Average events per T0 date:   1.65
Maximum events on one T0:     6

Events per T0 date:
1    88
2    25
3    13
4     5
5     2
6     3

EVENT CLUSTERING
Events sharing T0 with another event: 137
Percentage of events sharing T0: 60.89%

SANITY CHECKS
Duplicate event IDs: 0
T0 dates outside Bank Nifty data: 0
Events with T0 before earnings date: 0

Event calendar created: (225, 13)


In [9]:
# ============================================================
# CELL 4 — BUILD PRE-T0 INFORMATION SET
#
# IMPORTANT:
# Every feature below must be knowable BEFORE T0.
#
# T0 returns, T0 ranges, current-event actuals,
# current-event surprises and current-event reaction
# are NOT used as predictors.
# ============================================================


# ------------------------------------------------------------
# 1. Start with the event calendar
# ------------------------------------------------------------

pred_events = event_calendar.copy()

pred_events["t0_date"] = pd.to_datetime(
    pred_events["t0_date"]
)

pred_events["date"] = pd.to_datetime(
    pred_events["date"]
)


# ------------------------------------------------------------
# 2. Current earnings forecasts
#
# These are information available before the result.
# ------------------------------------------------------------

pred_events["eps_forecast_available"] = (
    pred_events["eps_forecast"].notna()
)

pred_events["revenue_forecast_available"] = (
    pred_events["revenue_forecast"].notna()
)


# ------------------------------------------------------------
# 3. Historical earnings behaviour
#
# VERY IMPORTANT:
# Shift BEFORE calculating historical statistics.
#
# Therefore the current event can never see its own
# surprise/reaction.
# ------------------------------------------------------------

pred_events = pred_events.sort_values(
    ["symbol", "date"]
).reset_index(drop=True)


pred_events["historical_eps_surprise"] = (
    pred_events
    .groupby("symbol")["eps_surprise"]
    .transform(
        lambda s:
        s.shift(1)
        .expanding()
        .mean()
    )
)

pred_events["historical_revenue_surprise"] = (
    pred_events
    .groupby("symbol")["revenue_surprise"]
    .transform(
        lambda s:
        s.shift(1)
        .expanding()
        .mean()
    )
)

pred_events["historical_abs_reaction"] = (
    pred_events
    .groupby("symbol")["reaction"]
    .transform(
        lambda s:
        s.shift(1)
        .abs()
        .expanding()
        .mean()
    )
)

pred_events["historical_reaction"] = (
    pred_events
    .groupby("symbol")["reaction"]
    .transform(
        lambda s:
        s.shift(1)
        .expanding()
        .mean()
    )
)

pred_events["previous_event_count"] = (
    pred_events
    .groupby("symbol")
    .cumcount()
)


# ------------------------------------------------------------
# 4. Constituent WEIGHT at T0
#
# Weight is structural information known before the move.
# ------------------------------------------------------------

pred_events = pred_events.merge(
    weights[
        [
            "date",
            "symbol",
            "weight"
        ]
    ],
    left_on=[
        "t0_date",
        "symbol"
    ],
    right_on=[
        "date",
        "symbol"
    ],
    how="left",
    suffixes=("", "_weight")
)

pred_events.drop(
    columns=["date_weight"],
    inplace=True,
    errors="ignore"
)


# ------------------------------------------------------------
# 5. PRE-T0 constituent movement
#
# Only T-3, T-2 and T-1.
# ------------------------------------------------------------

stock_pre = stocks[
    [
        "date",
        "symbol",
        "daily_return_pct",
        "intraday_range_pct"
    ]
].copy()


# Map each event to its T0.
stock_pre = stock_pre.merge(
    pred_events[
        [
            "event_id",
            "symbol",
            "t0_date"
        ]
    ],
    on="symbol",
    how="inner"
)

stock_pre = stock_pre[
    stock_pre["date"] < stock_pre["t0_date"]
]

stock_pre["days_before_t0"] = (
    stock_pre["t0_date"]
    - stock_pre["date"]
).dt.days


# Keep only the three trading observations immediately
# preceding T0.
stock_pre["trading_rank"] = (
    stock_pre
    .sort_values(["event_id", "date"])
    .groupby("event_id")
    .cumcount(ascending=False)
)

stock_pre = stock_pre[
    stock_pre["trading_rank"] < 3
].copy()


# Aggregate pre-event stock behaviour.
stock_pre_features = (
    stock_pre
    .groupby("event_id")
    .agg(
        stock_pre3_return=(
            "daily_return_pct",
            "sum"
        ),

        stock_pre3_mean_abs_return=(
            "daily_return_pct",
            lambda x: x.abs().mean()
        ),

        stock_pre3_max_abs_return=(
            "daily_return_pct",
            lambda x: x.abs().max()
        ),

        stock_pre3_mean_range=(
            "intraday_range_pct",
            "mean"
        ),

        stock_pre3_volatility=(
            "daily_return_pct",
            "std"
        ),

        stock_pre_observations=(
            "daily_return_pct",
            "count"
        ),
    )
    .reset_index()
)


pred_events = pred_events.merge(
    stock_pre_features,
    on="event_id",
    how="left"
)


# ------------------------------------------------------------
# 6. Bank Nifty PRE-T0 state
#
# Again, only observations BEFORE T0.
# ------------------------------------------------------------

bn_pre = banknifty[
    [
        "date",
        "daily_return_pct"
    ]
].copy()

bn_pre = bn_pre.sort_values("date")

# Historical rolling features.
bn_pre["bn_5d_return"] = (
    bn_pre["daily_return_pct"]
    .shift(1)
    .rolling(5, min_periods=5)
    .sum()
)

bn_pre["bn_5d_mean_abs_return"] = (
    bn_pre["daily_return_pct"]
    .shift(1)
    .abs()
    .rolling(5, min_periods=5)
    .mean()
)

bn_pre["bn_22d_volatility"] = (
    bn_pre["daily_return_pct"]
    .shift(1)
    .rolling(22, min_periods=15)
    .std()
)

bn_pre["bn_previous_day_return"] = (
    bn_pre["daily_return_pct"]
    .shift(1)
)

pred_events = pred_events.merge(
    bn_pre[
        [
            "date",
            "bn_5d_return",
            "bn_5d_mean_abs_return",
            "bn_22d_volatility",
            "bn_previous_day_return"
        ]
    ],
    left_on="t0_date",
    right_on="date",
    how="left"
)

pred_events.drop(
    columns=["date_y"],
    inplace=True,
    errors="ignore"
)

if "date_x" in pred_events.columns:
    pred_events.rename(
        columns={"date_x": "earnings_date"},
        inplace=True
    )


# ------------------------------------------------------------
# 7. Simultaneous earnings events
#
# This is known from the earnings calendar.
# ------------------------------------------------------------

events_per_t0 = (
    pred_events
    .groupby("t0_date")["event_id"]
    .nunique()
    .rename("events_on_t0")
)

pred_events = pred_events.merge(
    events_per_t0,
    on="t0_date",
    how="left"
)


# ------------------------------------------------------------
# 8. Build the DATE-LEVEL prediction dataset
#
# One Bank Nifty target date = one observation.
#
# Multiple constituent events on the same date are aggregated.
# ------------------------------------------------------------

prediction_features = (
    pred_events
    .groupby("t0_date")
    .agg(

        # Event structure
        event_count=(
            "event_id",
            "nunique"
        ),

        total_event_weight=(
            "weight",
            "sum"
        ),

        max_event_weight=(
            "weight",
            "max"
        ),

        # Current forecast information
        mean_eps_forecast=(
            "eps_forecast",
            "mean"
        ),

        mean_revenue_forecast=(
            "revenue_forecast",
            "mean"
        ),

        # Pre-event stock behaviour
        mean_stock_pre3_return=(
            "stock_pre3_return",
            "mean"
        ),

        mean_stock_pre3_abs_return=(
            "stock_pre3_mean_abs_return",
            "mean"
        ),

        max_stock_pre3_abs_return=(
            "stock_pre3_max_abs_return",
            "max"
        ),

        mean_stock_pre3_volatility=(
            "stock_pre3_volatility",
            "mean"
        ),

        # Historical earnings behaviour
        mean_historical_eps_surprise=(
            "historical_eps_surprise",
            "mean"
        ),

        mean_historical_revenue_surprise=(
            "historical_revenue_surprise",
            "mean"
        ),

        mean_historical_reaction=(
            "historical_reaction",
            "mean"
        ),

        mean_historical_abs_reaction=(
            "historical_abs_reaction",
            "mean"
        ),

        # Bank Nifty pre-event state
        bn_5d_return=(
            "bn_5d_return",
            "first"
        ),

        bn_5d_mean_abs_return=(
            "bn_5d_mean_abs_return",
            "first"
        ),

        bn_22d_volatility=(
            "bn_22d_volatility",
            "first"
        ),

        bn_previous_day_return=(
            "bn_previous_day_return",
            "first"
        ),
    )
    .reset_index()
)


# ------------------------------------------------------------
# 9. Add T0 TARGET
#
# This is deliberately added LAST.
# It must never be used as a feature.
# ------------------------------------------------------------

prediction_targets = (
    banknifty[
        [
            "date",
            "daily_return_pct"
        ]
    ]
    .rename(
        columns={
            "date": "t0_date",
            "daily_return_pct": "target_return"
        }
    )
)

prediction_features = prediction_features.merge(
    prediction_targets,
    on="t0_date",
    how="left"
)

prediction_features["target_abs_return"] = (
    prediction_features["target_return"].abs()
)

prediction_features["target_direction"] = np.select(
    [
        prediction_features["target_return"] > 0,
        prediction_features["target_return"] < 0,
    ],
    [
        "UP",
        "DOWN",
    ],
    default="FLAT"
)


# ------------------------------------------------------------
# 10. Final prediction dataset
# ------------------------------------------------------------

predictive_df = (
    prediction_features
    .sort_values("t0_date")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 11. Display structure
# ------------------------------------------------------------

print("=" * 70)
print("PRE-T0 PREDICTION DATASET")
print("=" * 70)

print(
    "Observations:",
    len(predictive_df)
)

print(
    "Date range:",
    predictive_df["t0_date"].min().date(),
    "→",
    predictive_df["t0_date"].max().date()
)

print("\nColumns:")
print(
    predictive_df.columns.tolist()
)

print("\nTarget distribution:")
print(
    predictive_df["target_direction"]
    .value_counts()
    .to_string()
)

print("\nMissingness:")
print(
    predictive_df.isna()
    .sum()
    .sort_values(ascending=False)
    .to_string()
)

PRE-T0 PREDICTION DATASET
Observations: 136
Date range: 2023-01-16 → 2026-08-06

Columns:
['t0_date', 'event_count', 'total_event_weight', 'max_event_weight', 'mean_eps_forecast', 'mean_revenue_forecast', 'mean_stock_pre3_return', 'mean_stock_pre3_abs_return', 'max_stock_pre3_abs_return', 'mean_stock_pre3_volatility', 'mean_historical_eps_surprise', 'mean_historical_revenue_surprise', 'mean_historical_reaction', 'mean_historical_abs_reaction', 'bn_5d_return', 'bn_5d_mean_abs_return', 'bn_22d_volatility', 'bn_previous_day_return', 'target_return', 'target_abs_return', 'target_direction']

Target distribution:
target_direction
UP      69
DOWN    67

Missingness:
max_event_weight                    12
mean_historical_abs_reaction         6
mean_historical_eps_surprise         6
mean_historical_reaction             6
mean_historical_revenue_surprise     6
bn_22d_volatility                    4
total_event_weight                   0
event_count                          0
t0_date            

In [10]:
# ============================================================
# CELL 5 — PREDICTIVE EDA: SIGNAL TEST
#
# Question 1:
# Can pre-T0 information tell us anything about
# Bank Nifty direction?
#
# Question 2:
# Can pre-T0 information tell us anything about
# Bank Nifty movement magnitude?
#
# NO T0 INFORMATION IS USED AS A FEATURE.
# ============================================================


# ------------------------------------------------------------
# 1. Candidate features
# ------------------------------------------------------------

candidate_features = [
    "event_count",
    "total_event_weight",
    "max_event_weight",

    "mean_eps_forecast",
    "mean_revenue_forecast",

    "mean_stock_pre3_return",
    "mean_stock_pre3_abs_return",
    "max_stock_pre3_abs_return",
    "mean_stock_pre3_volatility",

    "mean_historical_eps_surprise",
    "mean_historical_revenue_surprise",
    "mean_historical_reaction",
    "mean_historical_abs_reaction",

    "bn_5d_return",
    "bn_5d_mean_abs_return",
    "bn_22d_volatility",
    "bn_previous_day_return",
]


# ------------------------------------------------------------
# 2. Manual Spearman correlation
#
# We avoid scipy dependency.
# ------------------------------------------------------------

def spearman_corr(x, y):

    temp = pd.concat(
        [
            pd.Series(x).reset_index(drop=True),
            pd.Series(y).reset_index(drop=True),
        ],
        axis=1
    ).dropna()

    if len(temp) < 10:
        return np.nan

    return temp.iloc[:, 0].rank().corr(
        temp.iloc[:, 1].rank()
    )


# ------------------------------------------------------------
# 3. Calculate relationships
#
# Direction:
#   feature ↔ signed Bank Nifty return
#
# Magnitude:
#   feature ↔ absolute Bank Nifty return
# ------------------------------------------------------------

results = []

for feature in candidate_features:

    direction_rho = spearman_corr(
        predictive_df[feature],
        predictive_df["target_return"]
    )

    magnitude_rho = spearman_corr(
        predictive_df[feature],
        predictive_df["target_abs_return"]
    )

    results.append(
        {
            "feature": feature,
            "direction_rho": direction_rho,
            "magnitude_rho": magnitude_rho,
            "abs_direction_rho": (
                abs(direction_rho)
                if pd.notna(direction_rho)
                else np.nan
            ),
            "abs_magnitude_rho": (
                abs(magnitude_rho)
                if pd.notna(magnitude_rho)
                else np.nan
            ),
        }
    )


signal_results = (
    pd.DataFrame(results)
    .sort_values(
        "abs_magnitude_rho",
        ascending=False
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 4. Main signal table
# ------------------------------------------------------------

print("=" * 75)
print("PREDICTIVE EDA — PRE-T0 SIGNAL")
print("=" * 75)

print(
    signal_results[
        [
            "feature",
            "direction_rho",
            "magnitude_rho",
        ]
    ]
    .round(4)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 5. Strongest directional relationships
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STRONGEST DIRECTIONAL RELATIONSHIPS")
print("=" * 75)

direction_ranked = (
    signal_results
    .sort_values(
        "abs_direction_rho",
        ascending=False
    )
)

print(
    direction_ranked[
        [
            "feature",
            "direction_rho",
        ]
    ]
    .head(8)
    .round(4)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 6. Strongest magnitude relationships
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STRONGEST MAGNITUDE RELATIONSHIPS")
print("=" * 75)

magnitude_ranked = (
    signal_results
    .sort_values(
        "abs_magnitude_rho",
        ascending=False
    )
)

print(
    magnitude_ranked[
        [
            "feature",
            "magnitude_rho",
        ]
    ]
    .head(8)
    .round(4)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 7. UP vs DOWN comparison
#
# Correlation alone is not enough.
# We also check whether the pre-event environment differs
# between UP and DOWN event dates.
# ------------------------------------------------------------

direction_medians = (
    predictive_df
    .groupby("target_direction")[
        candidate_features
    ]
    .median()
    .T
)

direction_medians["UP_minus_DOWN"] = (
    direction_medians["UP"]
    - direction_medians["DOWN"]
)

print("\n" + "=" * 75)
print("UP vs DOWN — MEDIAN PRE-T0 FEATURES")
print("=" * 75)

print(
    direction_medians
    .round(4)
    .to_string()
)


# ------------------------------------------------------------
# 8. HIGH vs LOW MAGNITUDE
#
# Split at the median target magnitude.
# This is descriptive EDA, not a model.
# ------------------------------------------------------------

magnitude_cutoff = (
    predictive_df["target_abs_return"]
    .median()
)

predictive_df["magnitude_group"] = np.where(
    predictive_df["target_abs_return"]
    >= magnitude_cutoff,
    "HIGH",
    "LOW"
)

magnitude_medians = (
    predictive_df
    .groupby("magnitude_group")[
        candidate_features
    ]
    .median()
    .T
)

magnitude_medians["HIGH_minus_LOW"] = (
    magnitude_medians["HIGH"]
    - magnitude_medians["LOW"]
)

print("\n" + "=" * 75)
print(
    "HIGH vs LOW BANK NIFTY MOVEMENT"
    f" — median cutoff = {magnitude_cutoff:.4f}%"
)
print("=" * 75)

print(
    magnitude_medians
    .round(4)
    .to_string()
)


# ------------------------------------------------------------
# 9. Simple baseline for direction
#
# Because UP/DOWN is almost balanced, the naive baseline is
# simply the majority class.
# ------------------------------------------------------------

direction_counts = (
    predictive_df["target_direction"]
    .value_counts()
)

baseline_accuracy = (
    direction_counts.max()
    / direction_counts.sum()
)

print("\n" + "=" * 75)
print("DIRECTION BASELINE")
print("=" * 75)

print(
    f"Naive majority-class accuracy: "
    f"{baseline_accuracy:.2%}"
)

print(
    "UP:",
    direction_counts.get("UP", 0),
    "| DOWN:",
    direction_counts.get("DOWN", 0)
)


# ------------------------------------------------------------
# 10. Store outputs
# ------------------------------------------------------------

predictive_signal_results = signal_results.copy()

print("\n" + "=" * 75)
print("PREDICTIVE SIGNAL TEST COMPLETE")
print("=" * 75)

PREDICTIVE EDA — PRE-T0 SIGNAL
                         feature  direction_rho  magnitude_rho
                     event_count         0.0899         0.1282
      mean_stock_pre3_volatility         0.0543         0.0953
           mean_revenue_forecast         0.0292        -0.0844
        mean_historical_reaction        -0.0762        -0.0841
          mean_stock_pre3_return        -0.0045        -0.0785
       max_stock_pre3_abs_return         0.0463         0.0623
    mean_historical_abs_reaction        -0.0380        -0.0617
          bn_previous_day_return         0.0049        -0.0613
                    bn_5d_return        -0.0782        -0.0538
    mean_historical_eps_surprise        -0.0372        -0.0527
      mean_stock_pre3_abs_return         0.0311         0.0523
mean_historical_revenue_surprise        -0.1055         0.0515
           bn_5d_mean_abs_return         0.0922        -0.0363
               mean_eps_forecast        -0.0186        -0.0352
               bn_22d_vo

In [12]:
# ============================================================
# CELL 6 — PREDICTIVE EDA: BIVARIATE + MULTIVARIATE STRUCTURE
#
# Objectives:
# 1. Check relationships BETWEEN candidate predictors.
# 2. Identify redundant features.
# 3. Build a small multivariate baseline.
# 4. Compare it with naive baselines.
#
# IMPORTANT:
# This is still EDA, NOT final ML.
# We are testing whether multivariate predictive structure
# exists before committing to a modelling pipeline.
# ============================================================

from itertools import combinations


# ------------------------------------------------------------
# 1. Candidate features
# ------------------------------------------------------------

candidate_features = [
    "event_count",
    "total_event_weight",
    "max_event_weight",

    "mean_eps_forecast",
    "mean_revenue_forecast",

    "mean_stock_pre3_return",
    "mean_stock_pre3_abs_return",
    "max_stock_pre3_abs_return",
    "mean_stock_pre3_volatility",

    "mean_historical_eps_surprise",
    "mean_historical_revenue_surprise",
    "mean_historical_reaction",
    "mean_historical_abs_reaction",

    "bn_5d_return",
    "bn_5d_mean_abs_return",
    "bn_22d_volatility",
    "bn_previous_day_return",
]


# ------------------------------------------------------------
# 2. BIVARIATE FEATURE-TO-FEATURE RELATIONSHIPS
#
# Spearman is used because relationships need not be linear.
# ------------------------------------------------------------

feature_corr = predictive_df[
    candidate_features
].corr(
    method="spearman"
)

# Extract unique pairs only.
pair_results = []

for feature_a, feature_b in combinations(
    candidate_features,
    2
):
    rho = feature_corr.loc[
        feature_a,
        feature_b
    ]

    pair_results.append(
        {
            "feature_1": feature_a,
            "feature_2": feature_b,
            "spearman_rho": rho,
            "abs_rho": abs(rho),
        }
    )

pair_results = (
    pd.DataFrame(pair_results)
    .sort_values(
        "abs_rho",
        ascending=False
    )
    .reset_index(drop=True)
)

print("=" * 75)
print("PREDICTIVE BIVARIATE — FEATURE REDUNDANCY")
print("=" * 75)

print(
    pair_results[
        [
            "feature_1",
            "feature_2",
            "spearman_rho",
        ]
    ]
    .head(15)
    .round(4)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 3. Identify highly correlated feature pairs
#
# |rho| >= 0.70 is used only as a redundancy warning,
# NOT as an automatic deletion rule.
# ------------------------------------------------------------

high_corr_pairs = pair_results[
    pair_results["abs_rho"] >= 0.70
].copy()

print("\n" + "=" * 75)
print("HIGH-REDUNDANCY PAIRS — |rho| >= 0.70")
print("=" * 75)

if high_corr_pairs.empty:
    print("No feature pair crossed the 0.70 threshold.")
else:
    print(
        high_corr_pairs[
            [
                "feature_1",
                "feature_2",
                "spearman_rho",
            ]
        ]
        .round(4)
        .to_string(index=False)
    )


# ------------------------------------------------------------
# 4. Build a SMALL candidate feature set
#
# We deliberately avoid throwing every variable into the
# analysis. Each feature represents a different information
# category.
# ------------------------------------------------------------

multivariate_features = [
    # Event structure
    "event_count",
    "total_event_weight",

    # Constituent pre-event state
    "mean_stock_pre3_return",
    "mean_stock_pre3_abs_return",
    "mean_stock_pre3_volatility",

    # Historical earnings behaviour
    "mean_historical_eps_surprise",
    "mean_historical_revenue_surprise",
    "mean_historical_abs_reaction",

    # Index pre-event state
    "bn_5d_return",
    "bn_5d_mean_abs_return",
    "bn_22d_volatility",
]


# ------------------------------------------------------------
# 5. Prepare modelling matrix
#
# Median imputation is used ONLY for this exploratory test.
# It is performed inside each training fold below, so the
# test period does not influence training preprocessing.
# ------------------------------------------------------------

X = predictive_df[
    multivariate_features
].copy()

y_direction = (
    predictive_df["target_direction"]
    .map(
        {
            "DOWN": 0,
            "UP": 1,
        }
    )
)

y_magnitude = (
    predictive_df["target_abs_return"]
)


# ------------------------------------------------------------
# 6. Time-aware evaluation
#
# We deliberately do NOT random-split the data.
#
# Earlier dates -> training
# Later dates  -> testing
#
# This is closer to the real prediction problem.
# ------------------------------------------------------------

split_point = int(
    len(predictive_df) * 0.70
)

train_idx = np.arange(
    0,
    split_point
)

test_idx = np.arange(
    split_point,
    len(predictive_df)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_dir_train = y_direction.iloc[train_idx]
y_dir_test = y_direction.iloc[test_idx]

y_mag_train = y_magnitude.iloc[train_idx]
y_mag_test = y_magnitude.iloc[test_idx]


# ------------------------------------------------------------
# 7. Impute using TRAINING MEDIANS ONLY
# ------------------------------------------------------------

train_medians = X_train.median()

X_train = X_train.fillna(
    train_medians
)

X_test = X_test.fillna(
    train_medians
)


# ------------------------------------------------------------
# 8. Direction model
#
# Logistic regression is used only as a simple linear
# multivariate diagnostic.
# ------------------------------------------------------------

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
)

direction_model = LogisticRegression(
    max_iter=2000,
    random_state=42
)

direction_model.fit(
    X_train,
    y_dir_train
)

direction_pred = direction_model.predict(
    X_test
)

direction_accuracy = accuracy_score(
    y_dir_test,
    direction_pred
)

direction_balanced_accuracy = (
    balanced_accuracy_score(
        y_dir_test,
        direction_pred
    )
)


# ------------------------------------------------------------
# 9. Direction baseline
# ------------------------------------------------------------

majority_class = (
    y_dir_train
    .value_counts()
    .idxmax()
)

baseline_direction_pred = np.full(
    len(y_dir_test),
    majority_class
)

baseline_accuracy = accuracy_score(
    y_dir_test,
    baseline_direction_pred
)

baseline_balanced_accuracy = (
    balanced_accuracy_score(
        y_dir_test,
        baseline_direction_pred
    )
)


# ------------------------------------------------------------
# 10. Magnitude model
#
# Linear regression is used as a simple multivariate
# diagnostic—not as the final regression model.
# ------------------------------------------------------------

from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

magnitude_model = LinearRegression()

magnitude_model.fit(
    X_train,
    y_mag_train
)

magnitude_pred = magnitude_model.predict(
    X_test
)

magnitude_mae = mean_absolute_error(
    y_mag_test,
    magnitude_pred
)

magnitude_rmse = np.sqrt(
    mean_squared_error(
        y_mag_test,
        magnitude_pred
    )
)

magnitude_r2 = r2_score(
    y_mag_test,
    magnitude_pred
)


# ------------------------------------------------------------
# 11. Magnitude baseline
#
# Predict the TRAINING median for every test observation.
# ------------------------------------------------------------

magnitude_baseline_value = (
    y_mag_train.median()
)

magnitude_baseline_pred = np.full(
    len(y_mag_test),
    magnitude_baseline_value
)

baseline_mae = mean_absolute_error(
    y_mag_test,
    magnitude_baseline_pred
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_mag_test,
        magnitude_baseline_pred
    )
)

baseline_r2 = r2_score(
    y_mag_test,
    magnitude_baseline_pred
)


# ------------------------------------------------------------
# 12. Report multivariate diagnostic
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("MULTIVARIATE PREDICTIVE DIAGNOSTIC")
print("=" * 75)

print(
    f"Training observations: {len(train_idx)}"
)

print(
    f"Testing observations:  {len(test_idx)}"
)

print("\nDIRECTION")
print("-" * 75)

print(
    f"Naive baseline accuracy:       "
    f"{baseline_accuracy:.4f}"
)

print(
    f"Multivariate accuracy:         "
    f"{direction_accuracy:.4f}"
)

print(
    f"Naive baseline balanced acc.:  "
    f"{baseline_balanced_accuracy:.4f}"
)

print(
    f"Multivariate balanced acc.:    "
    f"{direction_balanced_accuracy:.4f}"
)


print("\nMAGNITUDE")
print("-" * 75)

print(
    f"Naive baseline MAE:            "
    f"{baseline_mae:.4f}%"
)

print(
    f"Multivariate MAE:              "
    f"{magnitude_mae:.4f}%"
)

print(
    f"Naive baseline RMSE:           "
    f"{baseline_rmse:.4f}%"
)

print(
    f"Multivariate RMSE:             "
    f"{magnitude_rmse:.4f}%"
)

print(
    f"Naive baseline R²:             "
    f"{baseline_r2:.4f}"
)

print(
    f"Multivariate R²:               "
    f"{magnitude_r2:.4f}"
)


# ------------------------------------------------------------
# 13. Inspect model coefficients
#
# These are descriptive diagnostics only.
# Do NOT interpret them as causal effects.
# ------------------------------------------------------------

direction_coefficients = (
    pd.Series(
        direction_model.coef_[0],
        index=multivariate_features
    )
    .sort_values(
        key=abs,
        ascending=False
    )
)

magnitude_coefficients = (
    pd.Series(
        magnitude_model.coef_,
        index=multivariate_features
    )
    .sort_values(
        key=abs,
        ascending=False
    )
)

print("\n" + "=" * 75)
print("DIRECTION MODEL — COEFFICIENTS")
print("=" * 75)

print(
    direction_coefficients
    .round(4)
    .to_string()
)

print("\n" + "=" * 75)
print("MAGNITUDE MODEL — COEFFICIENTS")
print("=" * 75)

print(
    magnitude_coefficients
    .round(4)
    .to_string()
)


# ------------------------------------------------------------
# 14. Store outputs
# ------------------------------------------------------------

predictive_bivariate_pairs = pair_results.copy()

predictive_multivariate_features = (
    multivariate_features.copy()
)

predictive_multivariate_summary = {
    "direction_baseline_accuracy": baseline_accuracy,
    "direction_model_accuracy": direction_accuracy,
    "direction_baseline_balanced_accuracy":
        baseline_balanced_accuracy,
    "direction_model_balanced_accuracy":
        direction_balanced_accuracy,
    "magnitude_baseline_mae": baseline_mae,
    "magnitude_model_mae": magnitude_mae,
    "magnitude_baseline_rmse": baseline_rmse,
    "magnitude_model_rmse": magnitude_rmse,
    "magnitude_baseline_r2": baseline_r2,
    "magnitude_model_r2": magnitude_r2,
}

print("\n" + "=" * 75)
print("PASS 2 COMPLETE")
print("=" * 75)

PREDICTIVE BIVARIATE — FEATURE REDUNDANCY
                       feature_1                    feature_2  spearman_rho
              total_event_weight             max_event_weight        0.9697
      mean_stock_pre3_abs_return    max_stock_pre3_abs_return        0.8389
      mean_stock_pre3_abs_return   mean_stock_pre3_volatility        0.8309
       max_stock_pre3_abs_return   mean_stock_pre3_volatility        0.7802
                max_event_weight        mean_revenue_forecast        0.6051
              total_event_weight            mean_eps_forecast        0.5973
                max_event_weight            mean_eps_forecast        0.5957
               mean_eps_forecast        mean_revenue_forecast        0.5630
              total_event_weight        mean_revenue_forecast        0.5625
           bn_5d_mean_abs_return            bn_22d_volatility        0.5292
          mean_stock_pre3_return                 bn_5d_return        0.5274
                     event_count           tot

In [13]:
# ============================================================
# CELL 7 — FINAL PREDICTIVE EDA
#
# T0 INFORMATION -> T+1 BANK NIFTY
#
# Prediction point:
# AFTER T0 MARKET CLOSE
#
# Therefore we may use:
#   - T0 Bank Nifty movement
#   - T0 constituent movement
#   - T0 weights
#   - actual earnings results
#   - earnings surprises
#   - earnings reaction
#   - pre-T0 information
#
# We MUST NOT use:
#   - T+1 return
#   - T+1 constituent movement
#   - any information generated during T+1
#
# This is the FINAL predictive EDA pass.
# ============================================================


# ------------------------------------------------------------
# 1. Build T0-level constituent event information
# ------------------------------------------------------------

t0_events = event_calendar.copy()

t0_events["t0_date"] = pd.to_datetime(
    t0_events["t0_date"]
)


# ------------------------------------------------------------
# 2. Add T0 stock movement
# ------------------------------------------------------------

stock_t0 = stocks[
    [
        "date",
        "symbol",
        "daily_return_pct",
        "intraday_range_pct",
    ]
].copy()

stock_t0 = stock_t0.rename(
    columns={
        "date": "t0_date",
        "daily_return_pct": "t0_stock_return",
        "intraday_range_pct": "t0_stock_range",
    }
)

t0_events = t0_events.merge(
    stock_t0,
    on=["t0_date", "symbol"],
    how="left"
)


# ------------------------------------------------------------
# 3. Add T0 weights
# ------------------------------------------------------------

weight_t0 = weights[
    [
        "date",
        "symbol",
        "weight",
    ]
].copy()

weight_t0 = weight_t0.rename(
    columns={
        "date": "t0_date"
    }
)

t0_events = t0_events.merge(
    weight_t0,
    on=["t0_date", "symbol"],
    how="left"
)


# ------------------------------------------------------------
# 4. Weighted T0 constituent movement
#
# Contribution approximation:
# weight (%) × stock return (%) / 100
# ------------------------------------------------------------

t0_events["t0_weighted_stock_return"] = (
    t0_events["weight"]
    * t0_events["t0_stock_return"]
    / 100
)


# ------------------------------------------------------------
# 5. Aggregate T0 event information to DATE level
#
# One Bank Nifty T+1 target = one observation.
# ------------------------------------------------------------

t0_date_features = (
    t0_events
    .groupby("t0_date")
    .agg(

        event_count=(
            "event_id",
            "nunique"
        ),

        total_event_weight=(
            "weight",
            "sum"
        ),

        max_event_weight=(
            "weight",
            "max"
        ),

        # Actual T0 constituent movement
        mean_t0_stock_return=(
            "t0_stock_return",
            "mean"
        ),

        mean_t0_stock_abs_return=(
            "t0_stock_return",
            lambda x: x.abs().mean()
        ),

        max_t0_stock_abs_return=(
            "t0_stock_return",
            lambda x: x.abs().max()
        ),

        mean_t0_stock_range=(
            "t0_stock_range",
            "mean"
        ),

        # Weighted constituent movement
        total_t0_weighted_return=(
            "t0_weighted_stock_return",
            "sum"
        ),

        max_abs_t0_weighted_return=(
            "t0_weighted_stock_return",
            lambda x: x.abs().max()
        ),

        # Actual earnings information
        mean_eps_surprise=(
            "eps_surprise",
            "mean"
        ),

        mean_revenue_surprise=(
            "revenue_surprise",
            "mean"
        ),

        mean_earnings_reaction=(
            "reaction",
            "mean"
        ),

        mean_abs_earnings_reaction=(
            "reaction",
            lambda x: x.abs().mean()
        ),

        # Actual reported results
        mean_eps_actual=(
            "eps_actual",
            "mean"
        ),

        mean_revenue_actual=(
            "revenue_actual",
            "mean"
        ),

        mean_eps_forecast=(
            "eps_forecast",
            "mean"
        ),

        mean_revenue_forecast=(
            "revenue_forecast",
            "mean"
        ),
    )
    .reset_index()
)


# ------------------------------------------------------------
# 6. Add T0 Bank Nifty information
# ------------------------------------------------------------

bn = banknifty[
    [
        "date",
        "daily_return_pct",
        "intraday_range_pct",
    ]
].copy()

bn = bn.rename(
    columns={
        "date": "t0_date",
        "daily_return_pct": "t0_bn_return",
        "intraday_range_pct": "t0_bn_range",
    }
)

t0_date_features = t0_date_features.merge(
    bn,
    on="t0_date",
    how="left"
)


# ------------------------------------------------------------
# 7. Create T+1 target
#
# Shift Bank Nifty return by one trading observation.
# ------------------------------------------------------------

bn_target = banknifty[
    [
        "date",
        "daily_return_pct",
    ]
].copy()

bn_target["t1_date"] = (
    bn_target["date"].shift(-1)
)

bn_target["t1_bn_return"] = (
    bn_target["daily_return_pct"].shift(-1)
)

bn_target = bn_target[
    [
        "date",
        "t1_date",
        "t1_bn_return",
    ]
].rename(
    columns={
        "date": "t0_date"
    }
)

t0_date_features = t0_date_features.merge(
    bn_target,
    on="t0_date",
    how="left"
)

t0_date_features["t1_bn_abs_return"] = (
    t0_date_features["t1_bn_return"].abs()
)

t0_date_features["t1_direction"] = np.select(
    [
        t0_date_features["t1_bn_return"] > 0,
        t0_date_features["t1_bn_return"] < 0,
    ],
    [
        "UP",
        "DOWN",
    ],
    default="FLAT"
)


# ------------------------------------------------------------
# 8. Add the PRE-T0 features from our previous analysis
#
# These are still valid because they were known before T0.
# ------------------------------------------------------------

pre_t0_features = predictive_df[
    [
        "t0_date",
        "mean_stock_pre3_return",
        "mean_stock_pre3_abs_return",
        "mean_stock_pre3_volatility",
        "mean_historical_eps_surprise",
        "mean_historical_revenue_surprise",
        "mean_historical_reaction",
        "mean_historical_abs_reaction",
        "bn_5d_return",
        "bn_5d_mean_abs_return",
        "bn_22d_volatility",
        "bn_previous_day_return",
    ]
].copy()

t0_date_features = t0_date_features.merge(
    pre_t0_features,
    on="t0_date",
    how="left"
)


# ------------------------------------------------------------
# 9. Final T+1 predictive dataset
# ------------------------------------------------------------

t1_predictive_df = (
    t0_date_features
    .dropna(
        subset=[
            "t1_bn_return",
            "t1_bn_abs_return",
        ]
    )
    .sort_values("t0_date")
    .reset_index(drop=True)
)


print("=" * 75)
print("FINAL T+1 PREDICTIVE DATASET")
print("=" * 75)

print(
    "Observations:",
    len(t1_predictive_df)
)

print(
    "T0 date range:",
    t1_predictive_df["t0_date"].min().date(),
    "→",
    t1_predictive_df["t0_date"].max().date()
)

print(
    "T+1 direction:"
)

print(
    t1_predictive_df["t1_direction"]
    .value_counts()
    .to_string()
)


# ------------------------------------------------------------
# 10. Candidate T+1 predictors
# ------------------------------------------------------------

t1_features = [

    # Event structure
    "event_count",
    "total_event_weight",

    # Earnings information known after T0
    "mean_eps_surprise",
    "mean_revenue_surprise",
    "mean_earnings_reaction",
    "mean_abs_earnings_reaction",

    # T0 constituent reaction
    "mean_t0_stock_return",
    "mean_t0_stock_abs_return",
    "max_t0_stock_abs_return",
    "mean_t0_stock_range",

    # Weighted constituent impact
    "total_t0_weighted_return",
    "max_abs_t0_weighted_return",

    # T0 Bank Nifty state
    "t0_bn_return",
    "t0_bn_range",

    # Pre-T0 context
    "mean_stock_pre3_return",
    "mean_stock_pre3_abs_return",
    "mean_stock_pre3_volatility",
    "mean_historical_eps_surprise",
    "mean_historical_revenue_surprise",
    "mean_historical_reaction",
    "mean_historical_abs_reaction",
    "bn_5d_return",
    "bn_5d_mean_abs_return",
    "bn_22d_volatility",
    "bn_previous_day_return",
]


# ------------------------------------------------------------
# 11. Manual Spearman
# ------------------------------------------------------------

def spearman_corr(x, y):

    temp = pd.concat(
        [
            pd.Series(x).reset_index(drop=True),
            pd.Series(y).reset_index(drop=True),
        ],
        axis=1
    ).dropna()

    if len(temp) < 10:
        return np.nan

    return temp.iloc[:, 0].rank().corr(
        temp.iloc[:, 1].rank()
    )


# ------------------------------------------------------------
# 12. Bivariate T+1 signal
# ------------------------------------------------------------

t1_signal = []

for feature in t1_features:

    direction_rho = spearman_corr(
        t1_predictive_df[feature],
        t1_predictive_df["t1_bn_return"]
    )

    magnitude_rho = spearman_corr(
        t1_predictive_df[feature],
        t1_predictive_df["t1_bn_abs_return"]
    )

    t1_signal.append(
        {
            "feature": feature,
            "direction_rho": direction_rho,
            "magnitude_rho": magnitude_rho,
            "abs_direction_rho": (
                abs(direction_rho)
                if pd.notna(direction_rho)
                else np.nan
            ),
            "abs_magnitude_rho": (
                abs(magnitude_rho)
                if pd.notna(magnitude_rho)
                else np.nan
            ),
        }
    )

t1_signal = (
    pd.DataFrame(t1_signal)
    .sort_values(
        "abs_magnitude_rho",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 75)
print("T+1 BIVARIATE PREDICTIVE SIGNAL")
print("=" * 75)

print(
    t1_signal[
        [
            "feature",
            "direction_rho",
            "magnitude_rho",
        ]
    ]
    .round(4)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 13. Time-aware multivariate diagnostic
#
# Use a compact feature set.
# ------------------------------------------------------------

t1_multivariate_features = [

    "event_count",
    "total_event_weight",

    "mean_eps_surprise",
    "mean_revenue_surprise",
    "mean_earnings_reaction",

    "mean_t0_stock_return",
    "mean_t0_stock_abs_return",
    "mean_t0_stock_range",

    "total_t0_weighted_return",

    "t0_bn_return",
    "t0_bn_range",

    "mean_stock_pre3_return",
    "mean_stock_pre3_volatility",

    "bn_5d_return",
    "bn_22d_volatility",
]


X = t1_predictive_df[
    t1_multivariate_features
].copy()

y_direction = (
    t1_predictive_df["t1_direction"]
    .map(
        {
            "DOWN": 0,
            "UP": 1,
        }
    )
)

y_magnitude = (
    t1_predictive_df["t1_bn_abs_return"]
)


# ------------------------------------------------------------
# 14. Time ordered split
# ------------------------------------------------------------

split_point = int(
    len(t1_predictive_df) * 0.70
)

train_idx = np.arange(
    0,
    split_point
)

test_idx = np.arange(
    split_point,
    len(t1_predictive_df)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_dir_train = y_direction.iloc[train_idx]
y_dir_test = y_direction.iloc[test_idx]

y_mag_train = y_magnitude.iloc[train_idx]
y_mag_test = y_magnitude.iloc[test_idx]


# ------------------------------------------------------------
# 15. Training-only imputation
# ------------------------------------------------------------

train_medians = X_train.median()

X_train = X_train.fillna(
    train_medians
)

X_test = X_test.fillna(
    train_medians
)


# ------------------------------------------------------------
# 16. Direction model
# ------------------------------------------------------------

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
)

direction_model = LogisticRegression(
    max_iter=3000,
    random_state=42
)

direction_model.fit(
    X_train,
    y_dir_train
)

direction_pred = direction_model.predict(
    X_test
)

direction_accuracy = accuracy_score(
    y_dir_test,
    direction_pred
)

direction_balanced_accuracy = (
    balanced_accuracy_score(
        y_dir_test,
        direction_pred
    )
)


# ------------------------------------------------------------
# 17. Direction baseline
# ------------------------------------------------------------

majority_class = (
    y_dir_train
    .value_counts()
    .idxmax()
)

baseline_direction_pred = np.full(
    len(y_dir_test),
    majority_class
)

baseline_direction_accuracy = (
    accuracy_score(
        y_dir_test,
        baseline_direction_pred
    )
)

baseline_direction_balanced_accuracy = (
    balanced_accuracy_score(
        y_dir_test,
        baseline_direction_pred
    )
)


# ------------------------------------------------------------
# 18. Magnitude model
# ------------------------------------------------------------

from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

magnitude_model = LinearRegression()

magnitude_model.fit(
    X_train,
    y_mag_train
)

magnitude_pred = magnitude_model.predict(
    X_test
)

magnitude_mae = mean_absolute_error(
    y_mag_test,
    magnitude_pred
)

magnitude_rmse = np.sqrt(
    mean_squared_error(
        y_mag_test,
        magnitude_pred
    )
)

magnitude_r2 = r2_score(
    y_mag_test,
    magnitude_pred
)


# ------------------------------------------------------------
# 19. Magnitude baseline
# ------------------------------------------------------------

magnitude_baseline_value = (
    y_mag_train.median()
)

magnitude_baseline_pred = np.full(
    len(y_mag_test),
    magnitude_baseline_value
)

baseline_mae = mean_absolute_error(
    y_mag_test,
    magnitude_baseline_pred
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_mag_test,
        magnitude_baseline_pred
    )
)

baseline_r2 = r2_score(
    y_mag_test,
    magnitude_baseline_pred
)


# ------------------------------------------------------------
# 20. Results
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("FINAL T+1 MULTIVARIATE PREDICTIVE DIAGNOSTIC")
print("=" * 75)

print(
    f"Training observations: {len(train_idx)}"
)

print(
    f"Testing observations:  {len(test_idx)}"
)

print("\nDIRECTION")
print("-" * 75)

print(
    f"Naive accuracy:              "
    f"{baseline_direction_accuracy:.4f}"
)

print(
    f"Multivariate accuracy:       "
    f"{direction_accuracy:.4f}"
)

print(
    f"Naive balanced accuracy:     "
    f"{baseline_direction_balanced_accuracy:.4f}"
)

print(
    f"Multivariate balanced acc.:  "
    f"{direction_balanced_accuracy:.4f}"
)

print("\nMAGNITUDE")
print("-" * 75)

print(
    f"Naive MAE:                   "
    f"{baseline_mae:.4f}%"
)

print(
    f"Multivariate MAE:            "
    f"{magnitude_mae:.4f}%"
)

print(
    f"Naive RMSE:                  "
    f"{baseline_rmse:.4f}%"
)

print(
    f"Multivariate RMSE:           "
    f"{magnitude_rmse:.4f}%"
)

print(
    f"Naive R²:                    "
    f"{baseline_r2:.4f}"
)

print(
    f"Multivariate R²:             "
    f"{magnitude_r2:.4f}"
)


# ------------------------------------------------------------
# 21. Model coefficients
# ------------------------------------------------------------

direction_coefficients = (
    pd.Series(
        direction_model.coef_[0],
        index=t1_multivariate_features
    )
    .sort_values(
        key=abs,
        ascending=False
    )
)

magnitude_coefficients = (
    pd.Series(
        magnitude_model.coef_,
        index=t1_multivariate_features
    )
    .sort_values(
        key=abs,
        ascending=False
    )
)

print("\n" + "=" * 75)
print("T+1 DIRECTION MODEL — COEFFICIENTS")
print("=" * 75)

print(
    direction_coefficients
    .round(4)
    .to_string()
)

print("\n" + "=" * 75)
print("T+1 MAGNITUDE MODEL — COEFFICIENTS")
print("=" * 75)

print(
    magnitude_coefficients
    .round(4)
    .to_string()
)


# ------------------------------------------------------------
# 22. Store final predictive EDA objects
# ------------------------------------------------------------

final_predictive_df = t1_predictive_df.copy()

final_t1_signal = t1_signal.copy()

print("\n" + "=" * 75)
print("PREDICTIVE EDA — FINAL PASS COMPLETE")
print("=" * 75)

FINAL T+1 PREDICTIVE DATASET
Observations: 136
T0 date range: 2023-01-16 → 2026-08-06
T+1 direction:
t1_direction
DOWN    82
UP      54

T+1 BIVARIATE PREDICTIVE SIGNAL
                         feature  direction_rho  magnitude_rho
      mean_abs_earnings_reaction        -0.1771         0.2268
mean_historical_revenue_surprise        -0.0624         0.2215
                    t0_bn_return        -0.0161        -0.2014
           bn_5d_mean_abs_return        -0.0086         0.1883
          mean_stock_pre3_return         0.0328        -0.1626
               bn_22d_volatility         0.0125         0.1586
      mean_stock_pre3_abs_return         0.0128         0.1579
               mean_eps_surprise         0.0279         0.1251
          bn_previous_day_return        -0.0132        -0.1211
                    bn_5d_return        -0.0504        -0.1046
    mean_historical_abs_reaction        -0.0094         0.0913
        total_t0_weighted_return        -0.0705         0.0894
    mean_his